In [1]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

from huggingface_hub import login
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE   : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.11.0+cu130
CUDA     : True
DEVICE   : cuda
GPU      : NVIDIA GeForce RTX 4090
VRAM     : 25.3 GB


In [2]:
MODEL_NAME      = "google/gemma-3-1b-it"
SAVE_PATH       = "./ccst_soft_prompt.pt"
NUM_SOFT_TOKENS = 20          # task_soft / per-class soft それぞれのトークン数
NUM_CLASSES     = 4           # AG News のクラス数
TEMPERATURE     = 0.07        # InfoNCE temperature
SEED            = 42
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
LABEL2ID = {label: idx for idx, label in enumerate(CLASS_NAMES)}

set_seed(SEED)

In [ ]:
HF_TOKEN = ""

hf_token = HF_TOKEN or os.getenv("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
print("HF_TOKEN 環境変数でログイン完了")

HF_TOKEN 環境変数でログイン完了


In [4]:
class CCSTSoftPrompt(nn.Module):
    """
    task_soft  : (NUM_SOFT_TOKENS, H)          タスク共通のsoft prompt
    class_soft : (NUM_CLASSES, NUM_SOFT_TOKENS, H)  クラスごとのsoft prompt
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.task_soft  = nn.Parameter(
            torch.randn(NUM_SOFT_TOKENS, hidden_size) * 0.01
        )
        self.class_soft = nn.Parameter(
            torch.randn(NUM_CLASSES, NUM_SOFT_TOKENS, hidden_size) * 0.01
        )

    def get_prefix(self, class_indices: torch.Tensor) -> torch.Tensor:
        """
        class_indices : (B,) long tensor
        return        : (B, 2*NUM_SOFT_TOKENS, H)
        """
        B    = class_indices.shape[0]
        task = self.task_soft.unsqueeze(0).expand(B, -1, -1)   # (B, S, H)
        cls  = self.class_soft[class_indices]                   # (B, S, H)
        return torch.cat([task, cls], dim=1)                    # (B, 2S, H)

In [5]:
def pool_text_hidden(
    model,
    prefix_embeds,       # (B, P, H) or None
    text_ids: torch.Tensor,   # (B, T)
) -> torch.Tensor:
    """
    text positions（prefix を除いた部分）の hidden states を mean-pool して返す。
    prefix_embeds が None のときはテキスト単体で forward する。
    """
    pad_id = model.config.pad_token_id
    # テキストのtoken embeddingを取得
    text_embeds = model.get_input_embeddings()(text_ids)    # (B, T, H)

    if prefix_embeds is not None:
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
        prefix_len    = prefix_embeds.shape[1]
    else:
        inputs_embeds = text_embeds
        prefix_len    = 0

    out    = model(inputs_embeds=inputs_embeds, output_hidden_states=True)
    hidden = out.hidden_states[-1]                          # (B, P+T, H)

    # text positions だけスライス
    text_hidden = hidden[:, prefix_len:, :]                 # (B, T, H)

    # padding を除いてmean pool
    mask   = (text_ids != pad_id).float().unsqueeze(-1)     # (B, T, 1)
    pooled = (text_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return pooled                                           # (B, H)

In [6]:
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")

# Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
pad_id = tokenizer.pad_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).to(DEVICE)

# LM のパラメータを全て固定
for p in model.parameters():
    p.requires_grad_(False)
model.eval()

hidden_size = model.config.hidden_size
print(f"Hidden size : {hidden_size}")

Device : cuda
Model  : google/gemma-3-1b-it


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Hidden size : 1152


In [7]:
soft_prompt = CCSTSoftPrompt(hidden_size).to(DEVICE)
soft_prompt.load_state_dict(torch.load("./ccst_infonce.pt", map_location=DEVICE))

<All keys matched successfully>

In [8]:
@torch.no_grad()
def get_next_topk(model, input_embeds, top_k=10):
    out = model(inputs_embeds=input_embeds, output_hidden_states=True)
    last_hidden = out.hidden_states[-1]                               # (B, seq, hidden)
    last_token_embeds = last_hidden[:, -1, :].unsqueeze(1)  # (B, 1, hidden)
    logits = model.lm_head(last_token_embeds)                 # (B, 1, vocab)
    token_prob = F.softmax(logits, dim=-1)  # (B, 1, vocab)
    top_k_prob, top_k_indices = torch.topk(token_prob, k=top_k, dim=-1)  # (B, 1, top_k)
    return top_k_prob, top_k_indices

In [48]:
@torch.no_grad()
def generate_text(model, input_ids, max_length=50, top_k=10, class_id=None):
    batch_size = input_ids.size(0)
    assert batch_size == 1, "この関数はバッチサイズ1でのみ動作します"

    model.eval()
    inputs_embeds = model.get_input_embeddings()(input_ids)   # (1, seq, hidden)
    if class_id is not None:
        prefix = soft_prompt.get_prefix(torch.tensor([class_id], device=input_ids.device))
        prefix = prefix.to(inputs_embeds.dtype)  # モデルのembeddingと同じdtypeに変換
    #    inputs_embeds = torch.cat([prefix, inputs_embeds], dim=1)

    n_generated_tokens = 0
    generated_ids = None
    while n_generated_tokens < max_length:
        top_k_prob, top_k_indices = get_next_topk(model, inputs_embeds, top_k)

        if class_id is not None:
            input_ids_expanded = input_ids.expand(top_k, -1)  # (top_k, seq)
            input_ids_expanded = torch.cat([input_ids_expanded, top_k_indices.squeeze().unsqueeze(1)], dim=1)  # (top_k, seq+1)
            
            text_only_rep = F.normalize(
                pool_text_hidden(model, None, input_ids_expanded), dim=-1
            )   # (top_k, H)
            soft_prompt_rep = F.normalize(
                pool_text_hidden(model, prefix.expand(top_k, -1, -1), input_ids_expanded), dim=-1
            )   # (top_k, H)
            sims = (text_only_rep * soft_prompt_rep).sum(-1).unsqueeze(0)   # (1, top_k)
            modified_top_k_prob = F.softmax(sims * 1e3, dim=-1)  # (1, top_k)
            next_token_id = torch.multinomial(modified_top_k_prob, num_samples=1)  # (1, 1)
        else:
            next_token_id = torch.multinomial(top_k_prob.squeeze(1), num_samples=1)  # (1, top_k) → (1, 1)
        next_token_id = top_k_indices[0, 0, next_token_id].reshape(1, 1)  # (1, 1)

        n_generated_tokens += 1
        if n_generated_tokens == 1:
            generated_ids = next_token_id
        elif next_token_id.item() == tokenizer.eos_token_id:
            break
        else:
            generated_ids = torch.cat([generated_ids, next_token_id], dim=1)  # (1, seq+1)

        input_ids = torch.cat([input_ids, generated_ids], dim=1)  # (1, seq+gen)
        next_input_embeds = model.get_input_embeddings()(generated_ids[:, -1:])  # (1, 1, hidden)
        inputs_embeds = torch.cat([inputs_embeds, next_input_embeds], dim=1)  # (1, seq+1, hidden)

    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text

In [49]:
prompt_text = "Today's news is about"
input_ids = torch.tensor([tokenizer.encode(prompt_text)], device=DEVICE)

for class_name in CLASS_NAMES:
    class_id = LABEL2ID[class_name]
    print("="*30)
    print(f"Class: {class_name}\n" + "-"*30)
    for _ in range(10):
        text = generate_text(
            model,
            input_ids=input_ids,
            max_length=50,
            top_k=10,
            class_id=class_id,
            #class_id=None,
        )
        generated_text = prompt_text + " " + text
        generated_text = " ".join(generated_text.split())
        print(generated_text)
        print("-"*30)
    print("\n\n")

Class: World
------------------------------
Today's news is about what to look when you see someone you think you might want to get to know better - focusing on how it affects you and your feelings about connection in your lives! We talk to therapist Dr Emily Hayes for a perspective to give it an added level,
------------------------------
Today's news is about what to look when you find yourself looking to hire or recruit in your organization or within an organisation to whom your organisation has an alliance to? Here in America, there have many great ways and opportunities you can leverage your recruitment strategies, but the right skills
------------------------------
Today's news is about what to look and how to approach to help with the mental illness of people who don support or are friends of their friends and the situation with depression during these months, * Let the first few paragraphs set the mood - This week, the shadows
------------------------------
Today's news is abou